# Stage 4 — Readability & Faithfulness Analysis

**Reads:** `outputs/explanations.json`  
**Writes:** `outputs/results.csv`

Computes per-explanation metrics:

| Metric | Description |
|--------|-------------|
| `flesch_reading_ease` | Higher = simpler text |
| `flesch_kincaid_grade` | Higher = more complex |
| `smog_index` | Estimated years of education needed |
| `lime_coverage` | Fraction of LIME features referenced in explanation |
| `ontology_hit_rate` | Fraction of features mapped to ≥1 ancestor |

> **Tip:** This notebook is fast — safe to re-run freely as you tweak display cells.

## 1. Imports

In [1]:
import pandas as pd

from config import ANALYSIS_RESULTS_PATH, EXPLANATIONS_PATH
from pipeline_helpers import (
    checkpoint_exists,
    lime_coverage,
    load_checkpoint,
    ontology_hit_rate,
    readability_metrics,
    save_checkpoint,
)

## 2. Configuration

In [2]:
# Set True to recompute even if results.csv already exists
FORCE_RERUN = True

## 3. Load Stage 3 output

In [3]:
if not EXPLANATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Explanations not found at '{EXPLANATIONS_PATH}'.\n"
        "Please run 03_llm.ipynb first."
    )
data = load_checkpoint(EXPLANATIONS_PATH)
print(f"Loaded {len(data)} explanations.")

[Checkpoint] Loaded 50 records ← 'outputs\explanations_beginner.json'
Loaded 50 explanations.


## 4. Compute metrics

In [4]:
rows = []
for item in data:
    explanation  = item.get("explanation", "")
    feature_data = item.get("feature_data", [])

    row = {
        "text":              item["text"][:80] + "…",
        "predicted_class":   item["predicted_class"],
        "confidence":        item["confidence"],
        "user_category":     item["user_category"],
        "ablation_mode":     item["ablation_mode"],
        "explanation":       explanation,
        "lime_coverage":     lime_coverage(explanation, feature_data),
        "ontology_hit_rate": ontology_hit_rate(feature_data),
        **readability_metrics(explanation),
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(f"✅ Metrics computed for {len(df)} explanations.")

✅ Metrics computed for 50 explanations.


## 5. Full results table

In [5]:
pd.set_option("display.max_colwidth", 60)
df[[
    "predicted_class", "user_category", "ablation_mode",
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "lime_coverage", "ontology_hit_rate"
]]

,predicted_class,user_category,ablation_mode,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
0,Digestive system diseases,BEGINNER,normal,35.106538,14.821795,16.322122,1.0000,1.0
1,Cardiovascular diseases,BEGINNER,normal,-11.430000,18.580000,17.122413,0.0000,0.0
2,Digestive system diseases,BEGINNER,normal,11.741916,21.186039,20.267339,1.0000,1.0
3,Cardiovascular diseases,BEGINNER,normal,21.714091,15.696061,16.728156,1.0000,1.0
4,Neoplasms,BEGINNER,normal,59.342045,9.081364,12.161745,1.0000,1.0
5,Neoplasms,BEGINNER,normal,43.483503,12.080000,13.559100,1.0000,1.0
6,General pathological conditions,BEGINNER,normal,-10.330000,18.675000,18.243606,0.0000,0.0
7,Neoplasms,BEGINNER,normal,40.037826,13.388696,14.068176,1.0000,1.0
8,General pathological conditions,BEGINNER,normal,5.552236,19.026835,20.581828,1.0000,1.0
9,Digestive system diseases,BEGINNER,normal,5.532500,16.462500,17.122413,0.0000,0.0


## 6. Summary — mean metrics by user category & ablation mode

In [6]:
metric_cols = [
    "flesch_reading_ease", "flesch_kincaid_grade",
    "smog_index", "lime_coverage", "ontology_hit_rate",
]
summary = (
    df.groupby(["user_category", "ablation_mode"])[metric_cols]
    .mean()
    .round(3)
)
summary

,,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
user_category,ablation_mode,,,,,
BEGINNER,normal,20.035,15.274,16.062,0.633,0.64


## 7. [Optional] Per-class breakdown

In [7]:
df.groupby("predicted_class")[metric_cols].mean().round(3)

,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
predicted_class,,,,,
Cardiovascular diseases,8.525,16.909,17.062,0.644,0.667
Digestive system diseases,26.896,14.921,16.121,0.727,0.727
General pathological conditions,7.532,16.671,17.165,0.500,0.500
Neoplasms,34.937,13.003,14.439,0.692,0.692
Nervous system diseases,21.198,14.509,14.943,0.333,0.333


## 8. [Optional] Read a specific explanation in full

In [8]:
# Change the index to read any explanation in full
IDX = 0

row = df.iloc[IDX]
print(f"Text     : {row['text']}")
print(f"Class    : {row['predicted_class']} ({row['confidence']})")
print(f"User     : {row['user_category']}")
print(f"Ablation : {row['ablation_mode']}")
print(f"\n── Explanation ────────────────────────────────────")
print(row["explanation"])

Text     : Normalization of ventilation/perfusion relationships after liver transplantation…
Class    : Digestive system diseases (0.5919)
User     : BEGINNER
Ablation : normal

── Explanation ────────────────────────────────────
assistant
The model predicted that the patient has digestive system diseases because it found the presence of the liver in the data. In the biomedical ontology, the liver is an organ located in the abdomen, which belongs to the digestive system. Therefore, if the model detected the liver in the dataset, it likely inferred that the patient might have conditions related to the digestive tract, such as gastritis, hepatitis, or other issues affecting this part of the body.


## 9. Save to CSV

In [9]:
df.to_csv(ANALYSIS_RESULTS_PATH, index=False)
print(f"✅ Saved {len(df)} rows → '{ANALYSIS_RESULTS_PATH}'")

✅ Saved 50 rows → 'outputs\results_beginner.csv'
